# Random Forest - LoL Rank Prediction

Trénování modelu Random Forest pro predikci ranku hráče.

In [ ]:
import sys
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.base import clone
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from preprocessing.cleaner import load_and_clean
from preprocessing.feature_engineering import build_player_dataset, AGGREGATED_FEATURE_NAMES, TIER_ORDER

print('Imports OK')

In [ ]:
# Configuration
MODEL_NAME = 'Random Forest'
SHORT_NAME = 'random_forest'
DESCRIPTION = 'Ensemble stromů s náhodným výběrem features'
COLOR = '#2ca02c'

DATASET_PATH = '../datasets/lol_rank_dataset.csv'
MODELS_DIR = '../models'

In [ ]:
# Load and prepare data
print('Loading data...')
raw_df = load_and_clean(DATASET_PATH)
player_df = build_player_dataset(raw_df)
print(f'Players: {len(player_df)}')

In [ ]:
# Prepare features
feature_names = [f for f in AGGREGATED_FEATURE_NAMES if f in player_df.columns]
X = player_df[feature_names].values
y = player_df['tier_encoded'].values
groups = player_df['puuid'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Define model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Cross-validation
gkf = GroupKFold(n_splits=5)
scores = cross_val_score(model, X_scaled, y, cv=gkf, groups=groups, scoring='accuracy')
cv_accuracy = scores.mean()
cv_std = scores.std()
print(f'CV Accuracy: {cv_accuracy:.4f} (+/- {cv_std:.4f})')

In [ ]:
# Train final model
model.fit(X_scaled, y)

# F1 score
gkf_split = list(gkf.split(X_scaled, y, groups))
train_idx, test_idx = gkf_split[0]
fold_model = clone(model)
fold_model.fit(X_scaled[train_idx], y[train_idx])
y_pred = fold_model.predict(X_scaled[test_idx])
f1 = f1_score(y[test_idx], y_pred, average='weighted')
print(f'F1 Score: {f1:.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y[test_idx], y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=TIER_ORDER, yticklabels=TIER_ORDER)
plt.title(f'{MODEL_NAME} - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance_df = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_})
importance_df = importance_df.sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'], color=COLOR)
plt.title(f'{MODEL_NAME} - Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
os.makedirs(MODELS_DIR, exist_ok=True)

with open(os.path.join(MODELS_DIR, f'model_{SHORT_NAME}.pkl'), 'wb') as f:
    pickle.dump(model, f)

meta = {
    'model_name': MODEL_NAME,
    'short_name': SHORT_NAME,
    'description': DESCRIPTION,
    'feature_names': feature_names,
    'tier_order': TIER_ORDER,
    'cv_accuracy': float(cv_accuracy),
    'cv_std': float(cv_std),
    'f1_score': float(f1),
    'num_players': len(player_df),
    'timestamp': datetime.now().isoformat()
}

with open(os.path.join(MODELS_DIR, f'model_{SHORT_NAME}_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'Model saved: model_{SHORT_NAME}.pkl')

# Random Forest - LoL Rank Prediction

**Model:** Random Forest Classifier  
**Popis:** Ensemble stromů s náhodným výběrem features  

Tento notebook trénuje Random Forest model pro predikci ranku hráčů v League of Legends.

## 1. Import knihoven

In [ ]:
import sys
import os
import json
import pickle
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.base import clone

warnings.filterwarnings('ignore')

# Přidání cesty k projektu
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

from Training.preprocessing.cleaner import load_and_clean
from Training.preprocessing.feature_engineering import (
    build_player_dataset, AGGREGATED_FEATURE_NAMES, TIER_ORDER,
)

print("Knihovny úspěšně načteny!")

## 2. Konfigurace

In [ ]:
# Konfigurace modelu
MODEL_NAME = "Random Forest"
MODEL_SHORT_NAME = "random_forest"
MODEL_DESCRIPTION = "Ensemble stromů s náhodným výběrem features"

# Cesty k souborům
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd())))
DATASET_PATH = os.path.join(BASE_DIR, 'Training', 'datasets', 'lol_rank_dataset.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'Training', 'models')

# Pokud cesty nefungují, zkusíme relativní
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = '../datasets/lol_rank_dataset.csv'
    MODELS_DIR = '../models'

print(f"Dataset: {DATASET_PATH}")
print(f"Models dir: {MODELS_DIR}")

## 3. Načtení a příprava dat

In [ ]:
# Načtení surových dat
print("Načítám data...")
raw_df = load_and_clean(DATASET_PATH)
print(f"Počet záznamů: {len(raw_df)}")
raw_df.head()

In [ ]:
# Vytvoření datasetu na úrovni hráčů
print("Vytvářím player-level dataset...")
player_df = build_player_dataset(raw_df)
print(f"Počet hráčů: {len(player_df)}")

# Distribuce tierů
print("\nDistribuce tierů:")
print(player_df['tier'].value_counts().sort_index())

In [ ]:
# Vizualizace distribuce tierů
plt.figure(figsize=(10, 6))
tier_counts = player_df['tier'].value_counts().reindex(TIER_ORDER)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(TIER_ORDER)))
plt.bar(TIER_ORDER, tier_counts.values, color=colors)
plt.xlabel('Tier')
plt.ylabel('Počet hráčů')
plt.title('Distribuce hráčů podle tieru')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Příprava features

In [ ]:
# Výběr features
feature_names = [f for f in AGGREGATED_FEATURE_NAMES if f in player_df.columns]
print(f"Počet features: {len(feature_names)}")
print(f"Features: {feature_names}")

X = player_df[feature_names].values
y = player_df['tier_encoded'].values
groups = player_df['puuid'].values

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Škálování features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Data škálována pomocí StandardScaler")

## 5. Definice modelu

In [ ]:
# Vytvoření modelu
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print(f"Model: {MODEL_NAME}")
print(f"Parametry: {model.get_params()}")

## 6. Cross-Validation

In [ ]:
# GroupKFold cross-validation
gkf = GroupKFold(n_splits=5)

print("Provádím 5-fold cross-validation...")
cv_scores = cross_val_score(model, X_scaled, y, cv=gkf, groups=groups, scoring='accuracy')

print(f"\nCV Accuracy scores: {cv_scores}")
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# Vizualizace CV scores
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_scores, color='forestgreen', alpha=0.7)
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title(f'{MODEL_NAME} - Cross-Validation Scores')
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Trénování finálního modelu

In [ ]:
# Trénování na všech datech
print("Trénuji finální model na všech datech...")
model.fit(X_scaled, y)
print("Model natrénován!")

## 8. Evaluace modelu

In [ ]:
# Evaluace na jednom foldu
gkf_split = list(gkf.split(X_scaled, y, groups))
train_idx, test_idx = gkf_split[0]

fold_model = clone(model)
fold_model.fit(X_scaled[train_idx], y[train_idx])
y_pred = fold_model.predict(X_scaled[test_idx])
y_true = y[test_idx]

# Metriky
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Test Accuracy (fold 1): {accuracy:.4f}")
print(f"F1 Score (weighted): {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=TIER_ORDER))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=TIER_ORDER, yticklabels=TIER_ORDER)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'{MODEL_NAME} - Confusion Matrix')
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='forestgreen')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title(f'{MODEL_NAME} - Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(feature_importance.head(10))

## 10. Uložení modelu

In [ ]:
# Vytvoření adresáře pro modely
os.makedirs(MODELS_DIR, exist_ok=True)

# Uložení modelu
model_filename = f'model_{MODEL_SHORT_NAME}.pkl'
with open(os.path.join(MODELS_DIR, model_filename), 'wb') as f:
    pickle.dump(model, f)
print(f"Model uložen: {model_filename}")

# Uložení scaleru
scaler_path = os.path.join(MODELS_DIR, 'scaler.pkl')
if not os.path.exists(scaler_path):
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print("Scaler uložen: scaler.pkl")

In [ ]:
# Uložení metadat
meta = {
    'model_name': MODEL_NAME,
    'short_name': MODEL_SHORT_NAME,
    'description': MODEL_DESCRIPTION,
    'feature_names': feature_names,
    'tier_order': TIER_ORDER,
    'cv_accuracy': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'f1_score': float(f1),
    'num_players': len(player_df),
    'timestamp': datetime.now().isoformat()
}

meta_filename = f'model_{MODEL_SHORT_NAME}_meta.json'
with open(os.path.join(MODELS_DIR, meta_filename), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"Metadata uložena: {meta_filename}")

## 11. Shrnutí

In [ ]:
print("="*60)
print(f"MODEL: {MODEL_NAME}")
print("="*60)
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"F1 Score (weighted): {f1:.4f}")
print(f"Počet features: {len(feature_names)}")
print(f"Počet hráčů v datasetu: {len(player_df)}")
print(f"\nSoubory uloženy:")
print(f"  - {model_filename}")
print(f"  - {meta_filename}")
print("="*60)